# Build a Logistic/Linear performance matrix on the AutoDP operator space

This notebook has one purpose: build the training performance matrix. It uses
Logistic Regression for classification and Linear Regression for
regression, with the 60/20/20 split used by the supplied notebook. The
preprocessing space is replaced by the six AutoDP families from the paper
(8,400 combinations including `none`) and 36 deterministic human-designed
reference pipelines are evaluated.

Methodological safeguards:

- preprocessing is fit on the train split only;
- validation and test rows are never removed;
- the full historical matrix may contain AutoDP test datasets, but the
  `reference` matrix always removes all 60 AutoDP IDs for later leakage-safe use;
- work is checkpointed and can optionally be sharded by `(dataset, pipeline)`.

This file is self-contained for Kaggle: the complete AutoDP implementation,
36 pipelines, 912 historical IDs and 60 holdout IDs are embedded directly.
Dataset tables and metadata are downloaded only from OpenML's read-only
GitLab/DataGit mirror; no OpenML API call or repository clone is required.


In [ ]:
# Kaggle setup. Re-running this cell is safe; no repository clone is needed.
from pathlib import Path
import os

if Path("/kaggle/working").exists():
    os.chdir("/kaggle/working")

print(f"Working directory: {Path.cwd()}")


In [ ]:
"""AutoDP operator space and leakage-safe preprocessing implementation.

The operator names follow Table 1 of the AutoDP paper.  ACORec uses a fixed,
explicit execution order for the reference performance matrix; operator-order
search, if desired, remains a separate ACORec concern.
"""
from __future__ import annotations

from collections import OrderedDict
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple
import zlib

import numpy as np
import pandas as pd

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.feature_selection import RFE
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import MinMaxScaler, StandardScaler


AUTODP_OPTIONS: "OrderedDict[str, List[str]]" = OrderedDict(
    [
        ("imputation", ["none", "random", "drop", "knn", "most_frequent", "em", "mice"]),
        ("encoding", ["none", "ordinal", "binary", "frequency", "catboost"]),
        ("normalization", ["none", "zscore", "minmax", "decimal_scale"]),
        ("feature_selection", ["none", "missing_ratio", "wrapper", "collinear", "tree_based"]),
        ("duplicate_removal", ["none", "exact", "approximate"]),
        ("outlier_removal", ["none", "zscore", "iqr", "lof"]),
    ]
)

# Cleaning precedes target-aware encoding; feature selection sees the final
# numeric representation.  This order is held constant for matrix comparability.
DEFAULT_AUTODP_ORDER: Tuple[str, ...] = (
    "imputation",
    "duplicate_removal",
    "outlier_removal",
    "encoding",
    "normalization",
    "feature_selection",
)

AUTODP_CLASSIFICATION_IDS: Tuple[int, ...] = (
    36, 728, 735, 737, 761, 803, 807, 816, 819, 871,
    1021, 1489, 31, 182, 183, 752, 833, 934, 979, 43723,
    32, 310, 1471, 43972, 179, 184, 734, 1053, 1461, 42493,
)

AUTODP_REGRESSION_IDS: Tuple[int, ...] = (
    189, 217, 225, 227, 503, 507, 529, 572, 573, 668,
    4551, 23516, 308, 516, 541, 558, 41021, 41700, 41928, 564,
    23515, 44019, 44031, 44057, 44066, 344, 41539, 41704, 44311, 45012,
)

AUTODP_60_IDS: Tuple[int, ...] = AUTODP_CLASSIFICATION_IDS + AUTODP_REGRESSION_IDS


def autodp_space_size(options: Mapping[str, Sequence[str]] = AUTODP_OPTIONS) -> int:
    """Return the Cartesian size of an AutoDP-style operator space."""
    return int(np.prod([len(values) for values in options.values()]))


def _pipeline(name: str, **overrides: str) -> Dict[str, str]:
    config = {stage: "none" for stage in AUTODP_OPTIONS}
    config.update(overrides)
    return {"name": name, **config}


def build_autodp_reference_pipelines() -> List[Dict[str, str]]:
    """Build 36 deterministic, human-designed reference pipelines.

    The design contains one all-none baseline, one pipeline for each of the 22
    non-none operators, and 13 deliberately diverse multi-stage pipelines.
    """
    pipelines = [_pipeline("autodp_00_baseline")]

    # One-factor-at-a-time coverage makes every operator independently visible
    # in the performance matrix.
    for stage, values in AUTODP_OPTIONS.items():
        for value in values:
            if value != "none":
                pipelines.append(_pipeline(f"autodp_ofat_{stage}_{value}", **{stage: value}))

    combined = [
        _pipeline("autodp_combo_01_basic", imputation="most_frequent", encoding="ordinal", normalization="zscore", feature_selection="missing_ratio", duplicate_removal="exact", outlier_removal="iqr"),
        _pipeline("autodp_combo_02_knn_binary", imputation="knn", encoding="binary", normalization="minmax", feature_selection="collinear", duplicate_removal="exact", outlier_removal="lof"),
        _pipeline("autodp_combo_03_mice_catboost", imputation="mice", encoding="catboost", normalization="zscore", feature_selection="tree_based", duplicate_removal="approximate", outlier_removal="zscore"),
        _pipeline("autodp_combo_04_em_frequency", imputation="em", encoding="frequency", normalization="decimal_scale", feature_selection="wrapper", duplicate_removal="exact", outlier_removal="iqr"),
        _pipeline("autodp_combo_05_random_ordinal", imputation="random", encoding="ordinal", normalization="zscore", feature_selection="collinear", duplicate_removal="approximate", outlier_removal="lof"),
        _pipeline("autodp_combo_06_drop_binary", imputation="drop", encoding="binary", normalization="minmax", feature_selection="tree_based", duplicate_removal="exact", outlier_removal="zscore"),
        _pipeline("autodp_combo_07_mf_frequency", imputation="most_frequent", encoding="frequency", normalization="decimal_scale", feature_selection="missing_ratio", duplicate_removal="approximate", outlier_removal="iqr"),
        _pipeline("autodp_combo_08_knn_catboost", imputation="knn", encoding="catboost", normalization="zscore", feature_selection="wrapper", outlier_removal="lof"),
        _pipeline("autodp_combo_09_mice_ordinal", imputation="mice", encoding="ordinal", normalization="minmax", feature_selection="collinear", duplicate_removal="exact"),
        _pipeline("autodp_combo_10_em_binary", imputation="em", encoding="binary", normalization="decimal_scale", feature_selection="missing_ratio", duplicate_removal="approximate", outlier_removal="zscore"),
        _pipeline("autodp_combo_11_random_frequency", imputation="random", encoding="frequency", normalization="zscore", feature_selection="tree_based", outlier_removal="iqr"),
        _pipeline("autodp_combo_12_drop_catboost", imputation="drop", encoding="catboost", normalization="minmax", feature_selection="wrapper", duplicate_removal="exact", outlier_removal="lof"),
        _pipeline("autodp_combo_13_mf_binary", imputation="most_frequent", encoding="binary", normalization="decimal_scale", feature_selection="collinear", duplicate_removal="approximate", outlier_removal="zscore"),
    ]
    pipelines.extend(combined)
    validate_autodp_reference_pipelines(pipelines)
    return pipelines


def validate_autodp_reference_pipelines(pipelines: Sequence[Mapping[str, str]]) -> None:
    """Fail fast if a reference pipeline is malformed or coverage regresses."""
    if len(pipelines) != 36:
        raise ValueError(f"Expected 36 AutoDP reference pipelines, got {len(pipelines)}")
    names = [config.get("name") for config in pipelines]
    if len(set(names)) != len(names):
        raise ValueError("AutoDP reference pipeline names must be unique")
    for config in pipelines:
        for stage, allowed in AUTODP_OPTIONS.items():
            if config.get(stage) not in allowed:
                raise ValueError(f"Invalid {stage}={config.get(stage)!r} in {config.get('name')}")
    for stage, allowed in AUTODP_OPTIONS.items():
        covered = {config[stage] for config in pipelines}
        missing = set(allowed) - covered
        if missing:
            raise ValueError(f"Reference pipelines do not cover {stage}: {sorted(missing)}")


def exclude_holdout_columns(
    performance_matrix: pd.DataFrame,
    holdout_ids: Iterable[int] = AUTODP_60_IDS,
) -> Tuple[pd.DataFrame, List[str]]:
    """Remove all holdout dataset columns before any ACORec training/retrieval."""
    forbidden = {f"D_{int(dataset_id)}" for dataset_id in holdout_ids}
    removed = [str(column) for column in performance_matrix.columns if str(column) in forbidden]
    return performance_matrix.drop(columns=removed), removed


class AutoDPPreprocessor:
    """Train-only fitted implementation of the AutoDP paper operator space.

    ``drop``, duplicate removal, and outlier removal may remove training rows.
    Validation/test rows are never removed, so prediction cardinality is stable.
    For ``drop``, held-out missing values use training-only fallback statistics.

    AutoDP's EM, MICE, approximate duplicate, and wrapper operators do not have
    exact scikit-learn equivalents.  Their deterministic approximations are
    documented in the corresponding methods below.
    """

    def __init__(
        self,
        config: Mapping[str, str],
        *,
        task_type: str,
        step_order: Optional[Sequence[str]] = None,
        random_state: int = 42,
        missing_ratio_threshold: float = 0.40,
        collinear_threshold: float = 0.95,
    ) -> None:
        self.config = dict(config)
        self.task_type = str(task_type)
        self.step_order = tuple(step_order or DEFAULT_AUTODP_ORDER)
        self.random_state = int(random_state)
        self.missing_ratio_threshold = float(missing_ratio_threshold)
        self.collinear_threshold = float(collinear_threshold)
        self.fitted = False

        for stage, allowed in AUTODP_OPTIONS.items():
            value = self.config.get(stage, "none")
            if value not in allowed:
                raise ValueError(f"Unsupported AutoDP operator: {stage}={value!r}")

        self.input_columns_: List[str] = []
        self.output_columns_: List[str] = []
        self.numeric_columns_: List[str] = []
        self.categorical_columns_: List[str] = []
        self.imputation_numeric_columns_: List[str] = []
        self.imputation_categorical_columns_: List[str] = []
        self.num_imputer_ = None
        self.cat_imputer_ = None
        self.fallback_num_imputer_ = None
        self.fallback_cat_imputer_ = None
        self.random_pools_: Dict[str, np.ndarray] = {}
        self.encoder_state_: Dict[str, object] = {}
        self.scaler_ = None
        self.scaler_columns_: List[str] = []
        self.decimal_scales_: Optional[pd.Series] = None
        self.selected_columns_: Optional[List[str]] = None

    @staticmethod
    def _frame(X: pd.DataFrame) -> pd.DataFrame:
        frame = pd.DataFrame(X).copy()
        frame.columns = frame.columns.astype(str)
        frame = frame.reset_index(drop=True)
        numeric = frame.select_dtypes(include=[np.number]).columns
        if len(numeric):
            frame[numeric] = frame[numeric].replace([np.inf, -np.inf], np.nan)
        return frame

    @staticmethod
    def _series(y: Optional[pd.Series]) -> Optional[pd.Series]:
        return None if y is None else pd.Series(y).reset_index(drop=True)

    @staticmethod
    def _safe_strings(series: pd.Series) -> pd.Series:
        return series.astype("string").fillna("__MISSING__")

    @staticmethod
    def _imputer_frame(imputer, frame: pd.DataFrame) -> pd.DataFrame:
        if frame.shape[1] == 0:
            return frame.copy()
        values = imputer.transform(frame)
        return pd.DataFrame(values, columns=frame.columns, index=frame.index)

    def fit_transform(self, X: pd.DataFrame, y: Optional[pd.Series] = None):
        X_work = self._frame(X)
        y_work = self._series(y)
        if y_work is not None and len(X_work) != len(y_work):
            raise ValueError("X and y must have the same length")
        self.input_columns_ = X_work.columns.tolist()
        self.numeric_columns_ = X_work.select_dtypes(include=[np.number]).columns.tolist()
        self.categorical_columns_ = [c for c in X_work.columns if c not in self.numeric_columns_]

        for step in self.step_order:
            if step == "imputation":
                X_work, y_work = self._fit_imputation(X_work, y_work)
            elif step == "duplicate_removal":
                X_work, y_work = self._fit_duplicate_removal(X_work, y_work)
            elif step == "outlier_removal":
                X_work, y_work = self._fit_outlier_removal(X_work, y_work)
            elif step == "encoding":
                X_work = self._fit_encoding(X_work, y_work)
            elif step == "normalization":
                X_work = self._fit_normalization(X_work)
            elif step == "feature_selection":
                X_work = self._fit_feature_selection(X_work, y_work)

        X_work = X_work.reset_index(drop=True)
        y_work = self._series(y_work)
        self.output_columns_ = X_work.columns.astype(str).tolist()
        self.fitted = True
        return X_work, y_work

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if not self.fitted:
            raise RuntimeError("fit_transform must be called before transform")
        X_work = self._frame(X)
        for column in self.input_columns_:
            if column not in X_work:
                X_work[column] = np.nan
        X_work = X_work[self.input_columns_]

        for step in self.step_order:
            if step == "imputation":
                X_work = self._transform_imputation(X_work)
            elif step in {"duplicate_removal", "outlier_removal"}:
                # Row-removal operators are training-only.
                continue
            elif step == "encoding":
                X_work = self._transform_encoding(X_work)
            elif step == "normalization":
                X_work = self._transform_normalization(X_work)
            elif step == "feature_selection":
                X_work = self._transform_feature_selection(X_work)

        for column in self.output_columns_:
            if column not in X_work:
                X_work[column] = np.nan
        return X_work[self.output_columns_].reset_index(drop=True)

    def _fit_fallback_imputers(self, X: pd.DataFrame) -> None:
        num = X[self.imputation_numeric_columns_].copy()
        cat = X[self.imputation_categorical_columns_].copy()
        if num.shape[1]:
            for column in num.columns[num.isna().all()]:
                num.loc[num.index[0], column] = 0.0
            self.fallback_num_imputer_ = SimpleImputer(strategy="median")
            self.fallback_num_imputer_.fit(num)
        if cat.shape[1]:
            for column in cat.columns[cat.isna().all()]:
                cat.loc[cat.index[0], column] = "__MISSING__"
            self.fallback_cat_imputer_ = SimpleImputer(strategy="most_frequent")
            self.fallback_cat_imputer_.fit(cat)

    def _apply_fallback_imputation(self, X: pd.DataFrame) -> pd.DataFrame:
        result = X.copy()
        if self.fallback_num_imputer_ is not None:
            result[self.imputation_numeric_columns_] = self._imputer_frame(
                self.fallback_num_imputer_, result[self.imputation_numeric_columns_]
            )
        if self.fallback_cat_imputer_ is not None:
            result[self.imputation_categorical_columns_] = self._imputer_frame(
                self.fallback_cat_imputer_, result[self.imputation_categorical_columns_]
            )
        return result

    def _random_fill(self, X: pd.DataFrame, *, fit: bool) -> pd.DataFrame:
        result = X.copy()
        for column in result.columns:
            if fit:
                pool = result[column].dropna().to_numpy()
                if len(pool) == 0:
                    pool = np.array([0.0 if column in self.numeric_columns_ else "__MISSING__"])
                self.random_pools_[column] = pool
            pool = self.random_pools_[column]
            missing = result[column].isna().to_numpy()
            if missing.any():
                seed = self.random_state + zlib.crc32(column.encode("utf-8"))
                rng = np.random.RandomState(seed & 0xFFFFFFFF)
                result.loc[missing, column] = rng.choice(pool, size=int(missing.sum()), replace=True)
        return result

    def _fit_imputation(self, X: pd.DataFrame, y: Optional[pd.Series]):
        method = self.config.get("imputation", "none")
        self.imputation_numeric_columns_ = X.select_dtypes(include=[np.number]).columns.tolist()
        self.imputation_categorical_columns_ = [
            column for column in X.columns if column not in self.imputation_numeric_columns_
        ]
        if method == "none":
            return X, y
        self._fit_fallback_imputers(X)
        if method == "drop":
            keep = ~X.isna().any(axis=1)
            if int(keep.sum()) < 2:
                raise ValueError("drop imputation left fewer than two training rows")
            return X.loc[keep].reset_index(drop=True), None if y is None else y.loc[keep].reset_index(drop=True)
        if method == "random":
            return self._random_fill(X, fit=True), y

        result = X.copy()
        num = result[self.imputation_numeric_columns_].copy()
        cat = result[self.imputation_categorical_columns_].copy()
        if num.shape[1]:
            for column in num.columns[num.isna().all()]:
                num.loc[num.index[0], column] = 0.0
            if method == "knn":
                self.num_imputer_ = KNNImputer(n_neighbors=max(1, min(5, len(X) - 1)))
            elif method == "em":
                # Deterministic iterative conditional expectation is the closest
                # stable sklearn analogue to EM imputation.
                self.num_imputer_ = IterativeImputer(max_iter=10, sample_posterior=False, random_state=self.random_state)
            elif method == "mice":
                self.num_imputer_ = IterativeImputer(max_iter=10, sample_posterior=True, random_state=self.random_state)
            else:
                self.num_imputer_ = SimpleImputer(strategy="most_frequent")
            result[self.imputation_numeric_columns_] = pd.DataFrame(
                self.num_imputer_.fit_transform(num), columns=self.imputation_numeric_columns_, index=result.index
            )
        if cat.shape[1]:
            for column in cat.columns[cat.isna().all()]:
                cat.loc[cat.index[0], column] = "__MISSING__"
            self.cat_imputer_ = SimpleImputer(strategy="most_frequent")
            result[self.imputation_categorical_columns_] = pd.DataFrame(
                self.cat_imputer_.fit_transform(cat), columns=self.imputation_categorical_columns_, index=result.index
            )
        return result, y

    def _transform_imputation(self, X: pd.DataFrame) -> pd.DataFrame:
        method = self.config.get("imputation", "none")
        if method == "none":
            return X
        if method == "drop":
            return self._apply_fallback_imputation(X)
        if method == "random":
            return self._random_fill(X, fit=False)
        result = X.copy()
        if self.num_imputer_ is not None:
            result[self.imputation_numeric_columns_] = self._imputer_frame(
                self.num_imputer_, result[self.imputation_numeric_columns_]
            )
        if self.cat_imputer_ is not None:
            result[self.imputation_categorical_columns_] = self._imputer_frame(
                self.cat_imputer_, result[self.imputation_categorical_columns_]
            )
        return result

    def _fit_duplicate_removal(self, X: pd.DataFrame, y: Optional[pd.Series]):
        method = self.config.get("duplicate_removal", "none")
        if method == "none" or len(X) < 2:
            return X, y
        if method == "exact":
            duplicate = X.duplicated(keep="first")
        else:
            # Scalable approximation: quantize numeric values to 1% of their
            # training range and normalize string representation before hashing.
            signature = pd.DataFrame(index=X.index)
            numeric = X.select_dtypes(include=[np.number]).columns
            categorical = [c for c in X.columns if c not in numeric]
            for column in numeric:
                values = pd.to_numeric(X[column], errors="coerce")
                low, high = values.min(), values.max()
                scale = high - low
                signature[column] = ((values - low) / scale).round(2) if pd.notna(scale) and scale > 0 else 0.0
                signature[column] = signature[column].fillna(-999.0)
            for column in categorical:
                signature[column] = self._safe_strings(X[column]).str.strip().str.lower()
            duplicate = signature.duplicated(keep="first")
        keep = ~duplicate
        return X.loc[keep].reset_index(drop=True), None if y is None else y.loc[keep].reset_index(drop=True)

    @staticmethod
    def _numeric_detector_frame(X: pd.DataFrame) -> pd.DataFrame:
        numeric = X.select_dtypes(include=[np.number]).copy()
        if numeric.shape[1] == 0:
            return numeric
        numeric = numeric.replace([np.inf, -np.inf], np.nan)
        return numeric.fillna(numeric.median()).fillna(0.0)

    def _fit_outlier_removal(self, X: pd.DataFrame, y: Optional[pd.Series]):
        method = self.config.get("outlier_removal", "none")
        detector = self._numeric_detector_frame(X)
        if method == "none" or detector.shape[1] == 0 or len(detector) < 3:
            return X, y
        if method == "zscore":
            std = detector.std(ddof=0).replace(0.0, 1.0)
            keep = ((detector - detector.mean()).abs().div(std) <= 3.0).all(axis=1)
        elif method == "iqr":
            q1, q3 = detector.quantile(0.25), detector.quantile(0.75)
            iqr = q3 - q1
            lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
            varying = iqr > 0
            keep = ((detector.loc[:, varying] >= lower[varying]) & (detector.loc[:, varying] <= upper[varying])).all(axis=1)
        else:
            neighbors = max(2, min(20, len(detector) - 1))
            keep = pd.Series(LocalOutlierFactor(n_neighbors=neighbors).fit_predict(detector) == 1, index=X.index)
        if int(keep.sum()) < max(2, int(0.20 * len(X))):
            # Avoid pathological operators deleting almost the entire training set.
            keep = pd.Series(True, index=X.index)
        return X.loc[keep].reset_index(drop=True), None if y is None else y.loc[keep].reset_index(drop=True)

    def _fit_encoding(self, X: pd.DataFrame, y: Optional[pd.Series]) -> pd.DataFrame:
        method = self.config.get("encoding", "none")
        categorical = X.select_dtypes(exclude=[np.number]).columns.tolist()
        self.encoder_state_ = {"method": method, "columns": categorical}
        if method == "none" or not categorical:
            return X
        numeric = X.drop(columns=categorical).reset_index(drop=True)
        encoded = self._encode_categorical(X[categorical].reset_index(drop=True), y, fit=True)
        return pd.concat([numeric, encoded], axis=1)

    def _transform_encoding(self, X: pd.DataFrame) -> pd.DataFrame:
        method = str(self.encoder_state_.get("method", "none"))
        categorical = list(self.encoder_state_.get("columns", []))
        if method == "none" or not categorical:
            return X
        numeric = X.drop(columns=categorical).reset_index(drop=True)
        encoded = self._encode_categorical(X[categorical].reset_index(drop=True), None, fit=False)
        return pd.concat([numeric, encoded], axis=1)

    def _encode_categorical(self, X_cat: pd.DataFrame, y: Optional[pd.Series], *, fit: bool) -> pd.DataFrame:
        method = str(self.encoder_state_["method"])
        parts: List[pd.DataFrame] = []
        if fit:
            self.encoder_state_["mappings"] = {}
        mappings = self.encoder_state_["mappings"]

        if method == "catboost":
            if fit and y is None:
                raise ValueError("CatBoost encoding requires y during fit")
            if fit:
                y_num = pd.to_numeric(pd.Series(y).reset_index(drop=True), errors="coerce")
                global_mean = float(y_num.mean())
                self.encoder_state_["global_mean"] = global_mean
            else:
                y_num = None
                global_mean = float(self.encoder_state_["global_mean"])
            alpha = 10.0
            for column in X_cat.columns:
                values = self._safe_strings(X_cat[column]).reset_index(drop=True)
                if fit:
                    sums: Dict[str, float] = {}
                    counts: Dict[str, int] = {}
                    ordered = np.empty(len(values), dtype=float)
                    rng = np.random.RandomState(self.random_state + zlib.crc32(column.encode("utf-8")))
                    for index in rng.permutation(len(values)):
                        key = str(values.iloc[index])
                        ordered[index] = (sums.get(key, 0.0) + alpha * global_mean) / (counts.get(key, 0) + alpha)
                        sums[key] = sums.get(key, 0.0) + float(y_num.iloc[index])
                        counts[key] = counts.get(key, 0) + 1
                    mappings[column] = {
                        key: (sums[key] + alpha * global_mean) / (counts[key] + alpha) for key in counts
                    }
                    encoded = ordered
                else:
                    encoded = values.map(mappings[column]).fillna(global_mean).to_numpy(dtype=float)
                parts.append(pd.DataFrame({f"{column}__catboost": encoded}))
            return pd.concat(parts, axis=1)

        for column in X_cat.columns:
            values = self._safe_strings(X_cat[column])
            if fit:
                unique = pd.Index(values.unique())
                if method in {"ordinal", "binary"}:
                    mappings[column] = {str(value): index + 1 for index, value in enumerate(unique)}
                elif method == "frequency":
                    mappings[column] = values.value_counts(normalize=True).to_dict()
            mapping = mappings[column]
            if method == "frequency":
                parts.append(pd.DataFrame({f"{column}__frequency": values.map(mapping).fillna(0.0).astype(float)}))
            else:
                codes = values.map(mapping).fillna(0).astype(int)
                if method == "ordinal":
                    parts.append(pd.DataFrame({f"{column}__ordinal": codes.astype(float)}))
                else:
                    if fit:
                        bits = max(1, int(np.ceil(np.log2(max(mapping.values(), default=0) + 1))))
                        self.encoder_state_.setdefault("bits", {})[column] = bits
                    bits = int(self.encoder_state_["bits"][column])
                    code_values = codes.to_numpy(dtype=np.int64)
                    parts.append(pd.DataFrame({
                        f"{column}__binary_{bit}": np.bitwise_and(
                            np.right_shift(code_values, bit), 1
                        ).astype(float)
                        for bit in range(bits)
                    }))
        return pd.concat(parts, axis=1)

    def _fit_normalization(self, X: pd.DataFrame) -> pd.DataFrame:
        method = self.config.get("normalization", "none")
        self.scaler_columns_ = X.select_dtypes(include=[np.number]).columns.tolist()
        if method == "none" or not self.scaler_columns_:
            return X
        result = X.copy()
        numeric = result[self.scaler_columns_]
        if method == "decimal_scale":
            max_abs = numeric.abs().max()
            powers = np.where(max_abs.fillna(0.0).to_numpy() > 0, np.floor(np.log10(max_abs.fillna(0.0).to_numpy())) + 1, 0)
            self.decimal_scales_ = pd.Series(np.power(10.0, powers), index=self.scaler_columns_).replace(0.0, 1.0)
            result[self.scaler_columns_] = numeric.divide(self.decimal_scales_, axis=1)
        else:
            self.scaler_ = StandardScaler() if method == "zscore" else MinMaxScaler()
            result[self.scaler_columns_] = pd.DataFrame(
                self.scaler_.fit_transform(numeric), columns=self.scaler_columns_, index=result.index
            )
        return result

    def _transform_normalization(self, X: pd.DataFrame) -> pd.DataFrame:
        method = self.config.get("normalization", "none")
        if method == "none" or not self.scaler_columns_:
            return X
        result = X.copy()
        if method == "decimal_scale":
            result[self.scaler_columns_] = result[self.scaler_columns_].divide(self.decimal_scales_, axis=1)
        else:
            result[self.scaler_columns_] = pd.DataFrame(
                self.scaler_.transform(result[self.scaler_columns_]),
                columns=self.scaler_columns_,
                index=result.index,
            )
        return result

    def _fit_feature_selection(self, X: pd.DataFrame, y: Optional[pd.Series]) -> pd.DataFrame:
        method = self.config.get("feature_selection", "none")
        if method == "none" or X.shape[1] <= 1:
            self.selected_columns_ = X.columns.tolist()
            return X
        if method == "missing_ratio":
            selected = X.columns[X.isna().mean() <= self.missing_ratio_threshold].tolist()
            self.selected_columns_ = selected or [X.isna().mean().idxmin()]
            return X[self.selected_columns_]

        numeric = X.select_dtypes(include=[np.number]).columns.tolist()
        categorical = [column for column in X.columns if column not in numeric]
        if len(numeric) <= 1:
            self.selected_columns_ = X.columns.tolist()
            return X
        model_X = self._numeric_detector_frame(X[numeric])

        if method == "collinear":
            correlation = model_X.corr().abs()
            upper = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))
            removed = {column for column in upper.columns if (upper[column] > self.collinear_threshold).any()}
            selected_numeric = [column for column in numeric if column not in removed]
        elif y is None:
            selected_numeric = numeric
        elif method == "tree_based":
            if self.task_type == "regression":
                model = ExtraTreesRegressor(n_estimators=64, random_state=self.random_state, n_jobs=-1)
            else:
                model = ExtraTreesClassifier(n_estimators=64, random_state=self.random_state, n_jobs=-1, class_weight="balanced")
            model.fit(model_X, y)
            order = np.argsort(model.feature_importances_)[::-1]
            keep_count = min(20, max(1, int(np.ceil(len(numeric) / 2))))
            selected_numeric = [numeric[index] for index in order[:keep_count]]
        else:
            # RFE is a practical sklearn analogue to AutoDP's wrapper evaluator.
            # Cap to 100 high-variance features to keep 900x36 matrix runs tractable.
            candidate = numeric
            if len(candidate) > 100:
                candidate = model_X.var().sort_values(ascending=False).head(100).index.tolist()
            estimator = Ridge(alpha=1.0) if self.task_type == "regression" else LogisticRegression(
                max_iter=500, solver="liblinear", random_state=self.random_state
            )
            keep_count = min(20, max(1, int(np.ceil(len(candidate) / 2))))
            selector = RFE(estimator=estimator, n_features_to_select=keep_count, step=0.2)
            selector.fit(model_X[candidate], y)
            selected_numeric = [column for column, keep in zip(candidate, selector.support_) if keep]

        self.selected_columns_ = selected_numeric + categorical
        if not self.selected_columns_:
            self.selected_columns_ = [X.columns[0]]
        return X[self.selected_columns_]

    def _transform_feature_selection(self, X: pd.DataFrame) -> pd.DataFrame:
        if self.selected_columns_ is None:
            return X
        for column in self.selected_columns_:
            if column not in X:
                X[column] = np.nan
        return X[self.selected_columns_]


__all__ = [
    "AUTODP_OPTIONS",
    "DEFAULT_AUTODP_ORDER",
    "AUTODP_CLASSIFICATION_IDS",
    "AUTODP_REGRESSION_IDS",
    "AUTODP_60_IDS",
    "AutoDPPreprocessor",
    "autodp_space_size",
    "build_autodp_reference_pipelines",
    "exclude_holdout_columns",
    "validate_autodp_reference_pipelines",
]


In [ ]:
# Imports and experiment controls
from __future__ import annotations

import json
import math
import os
import time
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.utils import shuffle

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

LINEAR_MODEL_CONFIG = {
    "logistic_C": 1.0,
    "logistic_max_iter": 1_000,
}

GITLAB_OPENML_ROOT = "https://gitlab.com/data/d/openml"
GITLAB_MAX_RETRIES = 5
GITLAB_RETRY_BASE_SECONDS = 3
DOWNLOAD_FAILURES = []

# Kaggle execution controls. Each run processes four consecutive shards.
RUN_MATRIX = True
CORPUS = "historical"          # "historical" or "autodp60"
NUM_JOB_SHARDS = 32
SHARDS_PER_RUN = 4
RUN_INDEX = 3                    # 0 <= index < ceil(NUM_JOB_SHARDS / SHARDS_PER_RUN)

start_shard = RUN_INDEX * SHARDS_PER_RUN
JOB_SHARD_INDICES = range(
    start_shard,
    min(start_shard + SHARDS_PER_RUN, NUM_JOB_SHARDS),
)
NUM_RUNS = math.ceil(NUM_JOB_SHARDS / SHARDS_PER_RUN)
if not 0 <= RUN_INDEX < NUM_RUNS:
    raise ValueError(f"RUN_INDEX must be between 0 and {NUM_RUNS - 1}")

MAX_SAMPLES_HISTORICAL = 5_000
MAX_SAMPLES_AUTODP60 = 100_000
OUTPUT_DIR = Path("/kaggle/working/autodp_matrix") if Path("/kaggle/working").exists() else Path("outputs/autodp_matrix")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_CACHE_DIR = (
    Path("/kaggle/temp/openml_gitlab_cache")
    if Path("/kaggle/temp").exists()
    else OUTPUT_DIR / "openml_gitlab_cache"
)
DATASET_CACHE_DIR.mkdir(parents=True, exist_ok=True)

pipeline_configs = build_autodp_reference_pipelines()
options = AUTODP_OPTIONS  # same variable name as the supplied notebook

print(f"AutoDP space size: {autodp_space_size():,}")
print(f"Reference pipelines: {len(pipeline_configs)}")
print(f"AutoDP holdout IDs: {len(AUTODP_60_IDS)}")
print(f"Run {RUN_INDEX + 1}/{NUM_RUNS}; shards: {list(JOB_SHARD_INDICES)}")
print(f"Dataset backend: GitLab/DataGit mirror ({GITLAB_OPENML_ROOT})")


In [ ]:
# Exact historical OpenML IDs are embedded so this notebook is standalone.
historical_dataset_ids = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 20, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 59, 60, 61, 62, 70, 71, 72, 73, 74, 75, 76, 77, 78, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 171, 172, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 195, 210, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 271, 272, 273, 274, 275, 276, 277, 278, 279, 285, 293, 300, 307, 310, 311, 312, 313, 316, 327, 328, 329, 333, 334, 335, 336, 337, 338, 339, 340, 342, 343, 346, 350, 351, 354, 357, 373, 375, 377, 378, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 443, 444, 446, 448, 450, 451, 452, 453, 454, 455, 457, 458, 459, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 472, 474, 475, 476, 477, 479, 480, 481, 488, 554, 679, 682, 683, 685, 694, 713, 714, 715, 716, 717, 718, 719, 720, 721, 722, 723, 724, 725, 726, 727, 728, 729, 730, 731, 732, 733, 734, 735, 736, 737, 738, 739, 740, 741, 742, 743, 744, 745, 746, 747, 748, 749, 750, 751, 752, 753, 754, 755, 756, 757, 758, 759, 760, 761, 762, 763, 764, 765, 766, 767, 768, 769, 770, 771, 772, 773, 774, 775, 776, 777, 778, 779, 780, 782, 783, 784, 785, 786, 787, 788, 789, 790, 791, 792, 793, 794, 795, 796, 797, 798, 799, 800, 801, 802, 803, 804, 805, 806, 807, 808, 810, 811, 812, 813, 814, 815, 816, 817, 818, 819, 820, 821, 823, 824, 825, 826, 827, 828, 829, 830, 831, 832, 833, 834, 835, 836, 837, 838, 839, 840, 841, 842, 843, 844, 845, 846, 847, 848, 849, 850, 851, 852, 853, 854, 855, 857, 858, 859, 860, 861, 862, 863, 864, 865, 866, 867, 868, 869, 870, 871, 873, 874, 875, 876, 877, 878, 879, 880, 881, 882, 884, 885, 886, 887, 888, 889, 890, 891, 892, 893, 894, 895, 896, 897, 898, 899, 900, 901, 902, 903, 904, 905, 906, 907, 908, 909, 910, 911, 912, 913, 914, 915, 916, 917, 918, 919, 920, 921, 922, 923, 924, 925, 926, 927, 928, 929, 930, 931, 932, 933, 934, 935, 936, 937, 938, 939, 940, 941, 942, 943, 944, 945, 946, 947, 949, 950, 951, 952, 953, 954, 955, 956, 957, 958, 959, 960, 961, 962, 963, 964, 965, 966, 967, 968, 969, 970, 971, 972, 973, 974, 975, 976, 977, 978, 979, 980, 981, 982, 983, 984, 985, 986, 987, 988, 989, 990, 991, 992, 993, 994, 995, 996, 997, 998, 999, 1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023, 1025, 1026, 1037, 1038, 1039, 1040, 1041, 1042, 1044, 1045, 1046, 1047, 1048, 1049, 1050, 1053, 1054, 1055, 1056, 1057, 1059, 1060, 1061, 1062, 1063, 1064, 1065, 1066, 1067, 1068, 1069, 1071, 1073, 1075, 1077, 1078, 1079, 1080, 1081, 1082, 1083, 1084, 1085, 1086, 1087, 1088, 1100, 1101, 1102, 1104, 1106, 1107, 1109, 1110, 1111, 1112, 1113, 1114, 1115, 1116, 1117, 1119, 1120, 1121, 1122, 1123, 1124, 1125, 1126, 1127, 1128, 1129, 1130, 1131, 1132, 1133, 1134, 1135, 1136, 1137, 1138, 1139, 1140, 1141, 1142, 1143, 1144, 1145, 1146, 1147, 1148, 1149, 1150, 1151, 1152, 1153, 1154, 1155, 1156, 1157, 1158, 1159, 1160, 1161, 1162, 1163, 1164, 1165, 1166, 1167, 1169, 1178, 1179, 1180, 1181, 1182, 1183, 1185, 1186, 1205, 1209, 1211, 1212, 1214, 1218, 1219, 1220, 1222, 1233, 1240, 1241, 1242, 1351, 1352, 1353, 1354, 1355, 1356, 1357, 1358, 1359, 1360, 1361, 1362, 1363, 1364, 1365, 1366, 1367, 1368, 1369, 1370, 1371, 1372, 1373, 1374, 1375, 1376, 1377, 1378, 1379, 1380, 1381, 1382, 1383, 1384, 1385, 1386, 1387, 1388, 1389, 1390, 1391, 1392, 1393, 1394, 1395, 1396, 1397, 1398, 1399, 1400, 1401, 1402, 1403, 1404, 1405, 1406, 1407, 1408, 1409, 1410, 1413, 1441, 1442, 1443, 1444, 1446, 1447, 1451, 1452, 1453, 1455, 1457, 1458, 1459, 1460, 1461, 1462, 1463, 1464, 1465, 1466, 1467, 1468, 1471, 1472, 1473, 1475, 1476, 1477, 1478, 1479, 1480, 1506, 1507, 1508, 1509, 1510, 1511, 1512, 1513, 1514, 1515, 1516, 1517, 1518, 1519, 1520, 1523, 1524, 1525, 1526, 1527, 1528, 1529, 1530, 1531, 1532, 1533, 1534, 1535, 1536, 1537, 1538, 1539, 1540, 1541, 1542, 1543, 1544, 1545, 1546, 1547, 1548, 1549, 1551, 1552, 1553, 1554, 1555, 1556, 1557, 1558, 1559, 1560, 1562, 1563, 1564, 1565, 1566, 1567, 1568, 1569, 1590, 1596, 1597, 4134, 4135, 4153, 4154, 4329, 4340, 4534, 4538, 4552, 6332, 23380, 23381, 23499, 23512, 23517, 40474, 40475, 40476, 40477, 40478, 40496, 40497, 40498, 40499, 40514, 40515, 40516, 40517, 40518, 40519, 40520, 40536, 40646, 40647, 40648, 40650, 40660, 40663, 40664, 40665, 40666, 40681, 40682, 40683, 40685, 40686, 40687, 40690, 40691, 40693, 40700, 40701, 40702, 40713, 40714, 40900, 40910, 40923, 40926, 40927, 40966, 40971, 40975, 40978, 40979, 41168, 41169, 41496, 41526, 41671]
autodp60_dataset_ids = list(AUTODP_60_IDS)
overlap_ids = sorted(set(historical_dataset_ids).intersection(AUTODP_60_IDS))
print(f"Embedded historical IDs: {len(historical_dataset_ids)} unique IDs")
print(f"Historical/AutoDP60 overlap: {len(overlap_ids)} IDs -> {overlap_ids}")

coverage = []
for stage, allowed in AUTODP_OPTIONS.items():
    counts = pd.Series([config[stage] for config in pipeline_configs]).value_counts()
    coverage.append({"stage": stage, **{value: int(counts.get(value, 0)) for value in allowed}})
display(pd.DataFrame(coverage).fillna("-"))
display(pd.DataFrame(pipeline_configs))


In [ ]:
# OpenML datasets are read only from the GitLab/DataGit mirror.
_GITLAB_SESSION = requests.Session()
_GITLAB_SESSION.headers.update({"User-Agent": "ACORec-AutoDP-matrix/1.0"})


def _gitlab_raw_url(dataset_id, relative_path):
    return (
        f"{GITLAB_OPENML_ROOT}/{int(dataset_id)}/-/raw/master/"
        f"{relative_path}"
    )


def _download_gitlab_file(dataset_id, relative_path, destination):
    destination = Path(destination)
    if destination.exists() and destination.stat().st_size > 0:
        return destination

    temporary = destination.with_suffix(destination.suffix + ".part")
    url = _gitlab_raw_url(dataset_id, relative_path)
    errors = []
    for attempt in range(1, GITLAB_MAX_RETRIES + 1):
        try:
            with _GITLAB_SESSION.get(
                url,
                stream=True,
                timeout=(20, 300),
                allow_redirects=True,
            ) as response:
                response.raise_for_status()
                with temporary.open("wb") as output:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            output.write(chunk)
            if not temporary.exists() or temporary.stat().st_size == 0:
                raise IOError(f"empty response from {url}")
            os.replace(temporary, destination)
            return destination
        except Exception as error:
            errors.append(f"attempt {attempt}: {type(error).__name__}: {error}")
            temporary.unlink(missing_ok=True)
            if attempt < GITLAB_MAX_RETRIES:
                wait_seconds = GITLAB_RETRY_BASE_SECONDS * (2 ** (attempt - 1))
                print(
                    f"  GitLab retry {attempt}/{GITLAB_MAX_RETRIES} "
                    f"for D_{dataset_id} in {wait_seconds}s"
                )
                time.sleep(wait_seconds)
    raise RuntimeError(" | ".join(errors))


def _metadata_attributes(value):
    if value is None:
        return []
    if isinstance(value, (list, tuple)):
        return [str(item).strip() for item in value if str(item).strip()]
    return [item.strip() for item in str(value).split(",") if item.strip()]


def download_gitlab_dataset(dataset_id):
    dataset_id = int(dataset_id)
    dataset_cache = DATASET_CACHE_DIR / str(dataset_id)
    dataset_cache.mkdir(parents=True, exist_ok=True)
    metadata_path = _download_gitlab_file(
        dataset_id,
        "dataset/metadata.json",
        dataset_cache / "metadata.json",
    )
    parquet_path = _download_gitlab_file(
        dataset_id,
        "dataset/tables/data.pq",
        dataset_cache / "data.pq",
    )

    with metadata_path.open("r", encoding="utf-8") as metadata_file:
        metadata = json.load(metadata_file)
    description = metadata.get("data_set_description", metadata)

    frame = pd.read_parquet(parquet_path)
    frame.columns = frame.columns.astype(str)
    targets = _metadata_attributes(description.get("default_target_attribute"))
    if len(targets) != 1:
        raise ValueError(
            f"expected one default target in GitLab metadata, got {targets}"
        )
    target = targets[0]
    if target not in frame.columns:
        raise KeyError(f"target {target!r} is absent from the Parquet columns")

    excluded = {target}
    excluded.update(_metadata_attributes(description.get("ignore_attribute")))
    excluded.update(_metadata_attributes(description.get("row_id_attribute")))
    feature_columns = [column for column in frame.columns if column not in excluded]
    return frame[feature_columns], frame[target], "gitlab-parquet"


def load_gitlab_dataset(dataset_id, *, is_autodp60=False):
    try:
        X, y, download_backend = download_gitlab_dataset(dataset_id)
        X, y = X.copy(), y.copy()
        X.columns = X.columns.astype(str)
        X = X.dropna(axis=1, how="all")
        valid_target = ~y.isna()
        X = X.loc[valid_target].reset_index(drop=True)
        y = y.loc[valid_target].reset_index(drop=True)

        known_regression = int(dataset_id) in set(AUTODP_REGRESSION_IDS)
        numeric_target = pd.api.types.is_numeric_dtype(y)
        task_type = "regression" if known_regression or (numeric_target and y.nunique() > 50) else "classification"

        if task_type == "classification":
            y = pd.Series(LabelEncoder().fit_transform(y.astype(str)))
            counts = y.value_counts()
            keep_classes = counts[counts >= 5].index
            keep = y.isin(keep_classes)
            X, y = X.loc[keep].reset_index(drop=True), y.loc[keep].reset_index(drop=True)
            if y.nunique() < 2:
                raise ValueError("fewer than two classes remain after rare-class filtering")
        else:
            y = pd.to_numeric(y, errors="coerce")
            keep = y.notna()
            X, y = X.loc[keep].reset_index(drop=True), y.loc[keep].reset_index(drop=True)

        max_samples = MAX_SAMPLES_AUTODP60 if is_autodp60 else MAX_SAMPLES_HISTORICAL
        if len(X) > max_samples:
            X, y = shuffle(X, y, n_samples=max_samples, random_state=RANDOM_STATE)
            X, y = X.reset_index(drop=True), pd.Series(y).reset_index(drop=True)
        if len(X) < 10:
            raise ValueError(f"only {len(X)} valid rows")

        print(f"Loaded D_{dataset_id}: shape={X.shape}, task={task_type}, via={download_backend}")
        return {"id": int(dataset_id), "name": f"D_{int(dataset_id)}", "X": X, "y": y, "task_type": task_type}
    except Exception as error:
        print(f"FAILED D_{dataset_id}: {type(error).__name__}: {error}")
        DOWNLOAD_FAILURES.append({
            "dataset_id": int(dataset_id),
            "error_type": type(error).__name__,
            "error": str(error),
        })
        pd.DataFrame(DOWNLOAD_FAILURES).to_csv(
            OUTPUT_DIR / "failed_gitlab_downloads.csv",
            index=False,
        )
        return None


In [ ]:
# Leakage-safe split and evaluation.
def split_train_val_test(X, y, task_type):
    stratify = y if task_type == "classification" else None
    try:
        X_train, X_temp, y_train, y_temp = train_test_split(
            X, y, test_size=0.40, random_state=RANDOM_STATE, stratify=stratify
        )
        temp_stratify = y_temp if task_type == "classification" else None
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=temp_stratify
        )
    except ValueError:
        X_train, X_temp, y_train, y_temp = train_test_split(
            X, y, test_size=0.40, random_state=RANDOM_STATE
        )
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE
        )
    return tuple(item.reset_index(drop=True) for item in (X_train, X_val, X_test, y_train, y_val, y_test))


def _linear_adapter(X_train):
    numeric = X_train.select_dtypes(include=[np.number]).columns.tolist()
    categorical = [column for column in X_train.columns if column not in numeric]
    transformers = []
    if numeric:
        transformers.append(("num", SimpleImputer(strategy="median"), numeric))
    if categorical:
        transformers.append(("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
        ]), categorical))
    return ColumnTransformer(transformers=transformers, remainder="drop")


def _evaluate_pipeline_linear(dataset, pipeline_config):
    X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
        dataset["X"], dataset["y"], dataset["task_type"]
    )
    preprocessor = AutoDPPreprocessor(
        pipeline_config, task_type=dataset["task_type"], random_state=RANDOM_STATE
    )
    X_train, y_train = preprocessor.fit_transform(X_train, y_train)
    X_test = preprocessor.transform(X_test)
    if len(X_train) < 2 or X_train.shape[1] == 0:
        return np.nan
    adapter = _linear_adapter(X_train)
    model = LinearRegression() if dataset["task_type"] == "regression" else LogisticRegression(
        C=LINEAR_MODEL_CONFIG["logistic_C"],
        max_iter=LINEAR_MODEL_CONFIG["logistic_max_iter"],
        random_state=RANDOM_STATE,
    )
    estimator = Pipeline([("adapter", adapter), ("model", model)])
    estimator.fit(X_train, y_train)
    prediction = estimator.predict(X_test)
    return float(r2_score(y_test, prediction) if dataset["task_type"] == "regression" else accuracy_score(y_test, prediction))


def evaluate_pipeline(dataset, pipeline_config):
    try:
        return _evaluate_pipeline_linear(dataset, pipeline_config)
    except Exception as error:
        print(f"  FAILED {pipeline_config['name']} on {dataset['name']}: {type(error).__name__}: {error}")
        return np.nan


In [ ]:
# Resumable work planning, part-matrix writing, and shard merging.
def make_job_shard(dataset_ids, configs, shard_index, num_shards):
    if not 0 <= shard_index < num_shards:
        raise ValueError("JOB_SHARD_INDEX must satisfy 0 <= index < NUM_JOB_SHARDS")
    jobs = [(dataset_id, config["name"]) for dataset_id in dataset_ids for config in configs]
    start = math.floor(len(jobs) * shard_index / num_shards)
    stop = math.floor(len(jobs) * (shard_index + 1) / num_shards)
    selected = jobs[start:stop]
    print(f"Shard {shard_index}/{num_shards}: jobs [{start}, {stop}) = {len(selected)} evaluations")
    grouped = defaultdict(list)
    for dataset_id, pipeline_name in selected:
        grouped[dataset_id].append(pipeline_name)
    return grouped


def _atomic_csv(frame, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary)
    os.replace(temporary, path)


def run_job_shard(dataset_ids, configs, *, corpus, shard_index, num_shards):
    grouped = make_job_shard(dataset_ids, configs, shard_index, num_shards)
    part_path = OUTPUT_DIR / f"{corpus}_autodp36.part_{shard_index:04d}_of_{num_shards:04d}.csv"
    columns = [f"D_{dataset_id}" for dataset_id in grouped]
    matrix = pd.DataFrame(index=[config["name"] for config in configs], columns=columns, dtype=float)
    if part_path.exists():
        saved = pd.read_csv(part_path, index_col=0)
        matrix.update(saved)
        print(f"Resuming {part_path}")

    config_by_name = {config["name"]: config for config in configs}
    for dataset_id, pipeline_names in grouped.items():
        column = f"D_{dataset_id}"
        pending = [name for name in pipeline_names if pd.isna(matrix.loc[name, column])]
        if not pending:
            continue
        dataset = load_gitlab_dataset(dataset_id, is_autodp60=(corpus == "autodp60"))
        if dataset is None:
            _atomic_csv(matrix, part_path)
            continue
        for pipeline_name in pending:
            print(f"{column} | {pipeline_name}")
            score = evaluate_pipeline(dataset, config_by_name[pipeline_name])
            matrix.loc[pipeline_name, column] = score
            print(f"  score={score}")
            _atomic_csv(matrix, part_path)
    return matrix, part_path


def merge_matrix_shards(part_paths, *, full_output, reference_output=None):
    merged = None
    conflicts = []
    for path in sorted(map(Path, part_paths)):
        part = pd.read_csv(path, index_col=0)
        if merged is None:
            merged = part.copy()
            continue
        merged = merged.reindex(index=merged.index.union(part.index), columns=merged.columns.union(part.columns))
        for row in part.index:
            for column in part.columns:
                value = part.loc[row, column]
                if pd.isna(value):
                    continue
                existing = merged.loc[row, column]
                if pd.notna(existing) and not np.isclose(float(existing), float(value), equal_nan=True):
                    conflicts.append((row, column, existing, value, str(path)))
                merged.loc[row, column] = value
    if merged is None:
        raise ValueError("No shard files supplied")
    if conflicts:
        raise ValueError(f"Conflicting shard values (first five): {conflicts[:5]}")
    merged = merged.reindex(index=[config["name"] for config in pipeline_configs])
    _atomic_csv(merged, full_output)
    if reference_output is not None:
        reference, removed = exclude_holdout_columns(merged)
        forbidden = {f"D_{dataset_id}" for dataset_id in AUTODP_60_IDS}
        assert forbidden.isdisjoint(reference.columns)
        _atomic_csv(reference, reference_output)
        print(f"Leakage guard removed {len(removed)} present AutoDP60 columns: {removed}")
    print(f"Merged matrix: {merged.shape}, missing cells={int(merged.isna().sum().sum())}")
    return merged


In [ ]:
# Execute several consecutive shards in this Kaggle run.
corpus_ids = historical_dataset_ids if CORPUS == "historical" else autodp60_dataset_ids
completed_paths = []

if RUN_MATRIX:
    for shard_index in JOB_SHARD_INDICES:
        print(f"\n{'=' * 70}")
        print(
            f"Running shard {shard_index}/{NUM_JOB_SHARDS - 1} "
            f"for RUN_INDEX={RUN_INDEX}"
        )
        print(f"{'=' * 70}")
        shard_matrix, shard_path = run_job_shard(
            corpus_ids,
            pipeline_configs,
            corpus=CORPUS,
            shard_index=shard_index,
            num_shards=NUM_JOB_SHARDS,
        )
        completed_paths.append(shard_path)
        print(f"Completed shard {shard_index}: {shard_path}")

    print("\nCompleted files in this Kaggle run:")
    for path in completed_paths:
        print(path)
else:
    for shard_index in JOB_SHARD_INDICES:
        make_job_shard(
            corpus_ids,
            pipeline_configs,
            shard_index,
            NUM_JOB_SHARDS,
        )
    print("Dry run only. Set RUN_MATRIX=True to execute.")


In [ ]:
# Merge after all shards have been copied into OUTPUT_DIR.
# This cell is also safe as a dry run while shards are incomplete.
expected_pattern = f"{CORPUS}_autodp36.part_*_of_{NUM_JOB_SHARDS:04d}.csv"
part_paths = sorted(OUTPUT_DIR.glob(expected_pattern))
print(f"Found {len(part_paths)}/{NUM_JOB_SHARDS} shard files")

if len(part_paths) == NUM_JOB_SHARDS:
    full_path = OUTPUT_DIR / f"{CORPUS}_performance_matrix_autodp36_full.csv"
    reference_path = OUTPUT_DIR / "training_performance_matrix_autodp36_reference.csv" if CORPUS == "historical" else None
    merged_matrix = merge_matrix_shards(
        part_paths,
        full_output=full_path,
        reference_output=reference_path,
    )
    print(f"Full matrix: {full_path}")
    if reference_path:
        print(f"Leakage-safe reference matrix: {reference_path}")
else:
    print("Merge skipped until every shard is present.")


## Required downstream rule

Keep both outputs: `*_full.csv` is the archival result over the whole historical
corpus, while `training_performance_matrix_autodp36_reference.csv` has every
present AutoDP test ID removed and is the safe input for later experiments.

The JSON representation of the same 36 pipelines is available at
`aco/pipeline_configs_autodp36.json`.
